# Wallet Investigation: 0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7dab0b33e852b0b1950aa8

Use the Sui JSON-RPC to collect all transactions involving the target wallet, summarise SUI inflows/outflows, and cross-check with known exchange addresses.


In [28]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence

import pandas as pd  # type: ignore
import requests
from datetime import datetime, timezone


In [29]:
RPC_URL = "https://fullnode.mainnet.sui.io:443"
REQUEST_TIMEOUT = 30
MAX_RETRIES = 5
RETRY_BACKOFF = 2.0  # seconds between retries, scaled by attempt
MAX_TX_PER_WALLET = 200
TARGET_ADDRESS = "0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7dab0b33e852b0b1950aa8"


In [30]:
def rpc_call(method: str, params: Optional[Sequence[object]] = None) -> dict:
    payload = {"jsonrpc": "2.0", "id": 1, "method": method, "params": params or []}
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.post(RPC_URL, json=payload, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            data = resp.json()
            if "error" in data:
                raise RuntimeError(data["error"])
            return data["result"]
        except Exception as exc:
            last_error = exc
            backoff = RETRY_BACKOFF * (attempt + 1)
            print(f"RPC call {method} failed (attempt {attempt+1}/{MAX_RETRIES}); retrying in {backoff:.1f}s")
            import time
            time.sleep(backoff)
    raise RuntimeError(f"RPC call failed after retries: {method}") from last_error


In [31]:
def fetch_transaction_digests(address: str, filter_key: str) -> List[str]:
    digests: List[str] = []
    cursor = None
    while True:
        result = rpc_call(
            "suix_queryTransactionBlocks",
            [
                {
                    "filter": {filter_key: address},
                    "options": {"showBalanceChanges": True, "showEvents": True},
                },
                cursor,
                100,
                True,
            ],
        )
        digests.extend(row["digest"] for row in result["data"])
        cursor = result.get("nextCursor")
        if not result["hasNextPage"] or len(digests) >= MAX_TX_PER_WALLET:
            break
    return digests


def fetch_transaction_block(digest: str) -> dict:
    return rpc_call(
        "sui_getTransactionBlock",
        [
            digest,
            {
                "showInput": True,
                "showEvents": True,
                "showBalanceChanges": True,
                "showEffects": True,
            },
        ],
    )


In [32]:
def gather_transfers(address: str) -> pd.DataFrame:
    digests = set(fetch_transaction_digests(address, "FromAddress") + fetch_transaction_digests(address, "ToAddress"))
    records = []
    for digest in digests:
        block = fetch_transaction_block(digest)
        ts_ms = block.get("timestampMs")
        ts = datetime.fromtimestamp(int(ts_ms) / 1000, timezone.utc) if ts_ms else None
        balance_changes = block.get("balanceChanges") or []
        for change in balance_changes:
            if change.get("coinType") != "0x2::sui::SUI":
                continue
            owner = change.get("owner", {})
            owner_addr = owner.get("AddressOwner")
            amount_str = change.get("amount") or "0"
            amount_mist = int(amount_str)
            amount_sui = amount_mist / 1_000_000_000
            records.append({
                "tx_digest": digest,
                "timestamp": ts.isoformat() if ts else "",
                "owner": owner_addr,
                "amount_sui": amount_sui,
                "direction": "out" if amount_sui < 0 else "in",
            })
    df = pd.DataFrame(records)
    return df.sort_values("timestamp") if not df.empty else df

transfers_df = gather_transfers(TARGET_ADDRESS)
print(f"Total transfer records: {len(transfers_df)}")
transfers_df.head()


Total transfer records: 475


,tx_digest,timestamp,owner,amount_sui,direction
138,67dbXbvhHnhKncaTVHHFCRvhQWg5RaCMyZEBESEC91pC,2025-04-09T19:22:26.322000+00:00,0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7d...,-86145.999802,out
137,67dbXbvhHnhKncaTVHHFCRvhQWg5RaCMyZEBESEC91pC,2025-04-09T19:22:26.322000+00:00,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...,86146.000000,in
227,92AmFyM48DRHuXaNRNbVveuhTMQChJcZcVoFrdcepcPx,2025-04-09T19:24:24.748000+00:00,0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7d...,-64619.001758,out
226,92AmFyM48DRHuXaNRNbVveuhTMQChJcZcVoFrdcepcPx,2025-04-09T19:24:24.748000+00:00,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...,64619.000000,in
429,36JB7CexJu61zTaDaqAqZ877sV3hjWkdHJujVFXuC2SQ,2025-04-09T19:38:26.374000+00:00,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...,152419.000000,in


In [33]:
if transfers_df.empty:
    raise RuntimeError("No SUI transfers found. Increase MAX_TX_PER_WALLET or check connectivity.")

incoming = transfers_df[transfers_df['direction'] == 'in']['amount_sui'].sum()
outgoing = transfers_df[transfers_df['direction'] == 'out']['amount_sui'].sum()
print('Total incoming SUI:', incoming)
print('Total outgoing SUI:', outgoing)
transfers_df


Total incoming SUI: 154128303.00857002
Total outgoing SUI: -154128303.35919237


,tx_digest,timestamp,owner,amount_sui,direction
138,67dbXbvhHnhKncaTVHHFCRvhQWg5RaCMyZEBESEC91pC,2025-04-09T19:22:26.322000+00:00,0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7d...,-86145.999802,out
137,67dbXbvhHnhKncaTVHHFCRvhQWg5RaCMyZEBESEC91pC,2025-04-09T19:22:26.322000+00:00,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...,86146.000000,in
227,92AmFyM48DRHuXaNRNbVveuhTMQChJcZcVoFrdcepcPx,2025-04-09T19:24:24.748000+00:00,0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7d...,-64619.001758,out
226,92AmFyM48DRHuXaNRNbVveuhTMQChJcZcVoFrdcepcPx,2025-04-09T19:24:24.748000+00:00,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...,64619.000000,in
429,36JB7CexJu61zTaDaqAqZ877sV3hjWkdHJujVFXuC2SQ,2025-04-09T19:38:26.374000+00:00,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...,152419.000000,in
...,...,...,...,...,...
132,7yUyApzQjeqsyobRtxnCtTVrqgF8cW9C8y4wf7onx1TU,2025-09-17T18:01:42.789000+00:00,0xe79de7eb11df6ee3ba7a37ee97aa7a771762f4a9e307...,-78000.001020,out
96,2DsErexY9iKgmnjA2jJPWuzxgPmZP3Ep6XqrumTgr1cf,2025-09-18T15:53:51.495000+00:00,0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7d...,-173410.998573,out
95,2DsErexY9iKgmnjA2jJPWuzxgPmZP3Ep6XqrumTgr1cf,2025-09-18T15:53:51.495000+00:00,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...,173411.000000,in
98,2EjgB4JHZSWjiwiYtinPWa2F4ysE72F7RrYMLahi1eeJ,2025-09-26T07:14:28.014000+00:00,0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7d...,-159479.001503,out


In [34]:
net_by_owner = transfers_df.groupby(['owner', 'direction'])['amount_sui'].sum().unstack(fill_value=0)
net_by_owner['net_sui'] = net_by_owner.get('in', 0) + net_by_owner.get('out', 0)
net_by_owner.sort_values('net_sui')


direction,in,out,net_sui
owner,,,
0x7f3b918653aacd4289b7e471aea588b34d497df0e1540c2d22eadacccd3239e4,0.000000e+00,-4.663256e+07,-4.663256e+07
0xa4da59e5ba0dabefd1958fc42b4cbf8f01e165d311542e5da54475b68a9c8d0d,0.000000e+00,-2.135636e+07,-2.135636e+07
0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7dab0b33e852b0b1950aa8,7.360195e+07,-8.052636e+07,-6.924411e+06
0xe79de7eb11df6ee3ba7a37ee97aa7a771762f4a9e307113fa14adabcbbe776e7,0.000000e+00,-5.613019e+06,-5.613019e+06
0xad78305b23f61fefb19002d400d9ad0448d184bcd731bacb8fe236c049c1400a,0.000000e+00,-1.030788e-02,-1.030788e-02
0x23a25a4598ffc60e68297789fd3536adfa91dd8f3038f9f73321b6315d356c11,0.000000e+00,-2.101856e-03,-2.101856e-03
0x80809c83575b43d7cb89ae02ea1ad0096ce9fcbc801bbc715dacb80bd229ff02,0.000000e+00,-2.060856e-03,-2.060856e-03
0xcf6ab2f79173e4489a0c873a671dc819b076bb9074404d383f9ebcee1851921e,0.000000e+00,-1.860856e-03,-1.860856e-03
0xbe9e8a62a74d1246924e10c46b3de0a22ec20998f66d1562c615d3ee846e922e,8.328400e+04,0.000000e+00,8.328400e+04


In [35]:
known_path = Path('/Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data/known_addresses.csv')
known_df = pd.read_csv(known_path)
known_wallets = set(known_df['wallet'])


In [36]:
def gather_neighbors(address: str) -> Dict[str, float]:
    digests = set(fetch_transaction_digests(address, 'FromAddress') + fetch_transaction_digests(address, 'ToAddress'))
    neighbors: Dict[str, float] = defaultdict(float)
    for digest in digests:
        block = fetch_transaction_block(digest)
        for change in block.get('balanceChanges') or []:
            if change.get('coinType') != '0x2::sui::SUI':
                continue
            owner = change.get('owner', {})
            owner_addr = owner.get('AddressOwner')
            if not owner_addr or owner_addr == address:
                continue
            amount = float(change.get('amount') or 0) / 1_000_000_000
            neighbors[owner_addr] += amount
        for event in block.get('events') or []:
            event_type = event.get('type', '').lower()
            if 'stake' in event_type:
                validator = event.get('parsedJson', {}).get('validator_address')
                if validator and validator != address:
                    neighbors.setdefault(validator, 0.0)
    return neighbors


In [37]:
def bfs_until_known(seed: str, known_wallets: set[str], max_depth: int = 4) -> dict:
    visited: set[str] = set([seed])
    frontier: list[tuple[str, int]] = [(seed, 0)]
    adjacency: dict[str, list[str]] = {}
    while frontier:
        node, depth = frontier.pop(0)
        if depth >= max_depth:
            continue
        neighbors = gather_neighbors(node).keys()
        adjacency[node] = list(neighbors)
        for nbr in neighbors:
            if nbr in known_wallets:
                visited.add(nbr)
                adjacency.setdefault(node, []).append(nbr)
                continue
            if nbr not in visited:
                visited.add(nbr)
                frontier.append((nbr, depth + 1))
    return adjacency

# Example usage after building known_labels
known_wallets = set(known_df['wallet'])
bfs_graph = bfs_until_known(TARGET_ADDRESS, known_wallets, max_depth=3)
bfs_graph


RPC call sui_getTransactionBlock failed (attempt 1/5); retrying in 2.0s
RPC call sui_getTransactionBlock failed (attempt 1/5); retrying in 2.0s
RPC call sui_getTransactionBlock failed (attempt 1/5); retrying in 2.0s
RPC call sui_getTransactionBlock failed (attempt 1/5); retrying in 2.0s
RPC call suix_queryTransactionBlocks failed (attempt 1/5); retrying in 2.0s
RPC call suix_queryTransactionBlocks failed (attempt 1/5); retrying in 2.0s
RPC call suix_queryTransactionBlocks failed (attempt 1/5); retrying in 2.0s
RPC call suix_queryTransactionBlocks failed (attempt 1/5); retrying in 2.0s
RPC call suix_queryTransactionBlocks failed (attempt 1/5); retrying in 2.0s
RPC call sui_getTransactionBlock failed (attempt 1/5); retrying in 2.0s
RPC call sui_getTransactionBlock failed (attempt 1/5); retrying in 2.0s
RPC call sui_getTransactionBlock failed (attempt 2/5); retrying in 4.0s
RPC call sui_getTransactionBlock failed (attempt 3/5); retrying in 6.0s
RPC call sui_getTransactionBlock failed (att

RuntimeError: RPC call failed after retries: sui_getTransactionBlock

In [38]:
# Link to known exchange wallets
known_path = Path('/Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data/known_addresses.csv')
known_df = pd.read_csv(known_path)
known_labels = {row.wallet: row.label for row in known_df.itertuples()}

def annotate(row):
    wallet = row.name
    label = known_labels.get(wallet, '')
    return pd.Series({
        'label': label,
        'out_sui': row.get('out', 0),
        'in_sui': row.get('in', 0),
        'net_sui': row.get('net_sui'),
    })

annotated = net_by_owner.apply(annotate, axis=1)
annotated.sort_values('net_sui')



,label,out_sui,in_sui,net_sui
owner,,,,
0x7f3b918653aacd4289b7e471aea588b34d497df0e1540c2d22eadacccd3239e4,,-4.663256e+07,0.000000e+00,-4.663256e+07
0xa4da59e5ba0dabefd1958fc42b4cbf8f01e165d311542e5da54475b68a9c8d0d,,-2.135636e+07,0.000000e+00,-2.135636e+07
0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7dab0b33e852b0b1950aa8,,-8.052636e+07,7.360195e+07,-6.924411e+06
0xe79de7eb11df6ee3ba7a37ee97aa7a771762f4a9e307113fa14adabcbbe776e7,,-5.613019e+06,0.000000e+00,-5.613019e+06
0xad78305b23f61fefb19002d400d9ad0448d184bcd731bacb8fe236c049c1400a,,-1.030788e-02,0.000000e+00,-1.030788e-02
0x23a25a4598ffc60e68297789fd3536adfa91dd8f3038f9f73321b6315d356c11,,-2.101856e-03,0.000000e+00,-2.101856e-03
0x80809c83575b43d7cb89ae02ea1ad0096ce9fcbc801bbc715dacb80bd229ff02,,-2.060856e-03,0.000000e+00,-2.060856e-03
0xcf6ab2f79173e4489a0c873a671dc819b076bb9074404d383f9ebcee1851921e,,-1.860856e-03,0.000000e+00,-1.860856e-03
0xbe9e8a62a74d1246924e10c46b3de0a22ec20998f66d1562c615d3ee846e922e,,0.000000e+00,8.328400e+04,8.328400e+04


In [39]:
# Save detailed transfers
output_dir = Path('/Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data/diagnostics')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'wallet_0xbe90db_transfers.csv'
transfers_df.to_csv(output_path, index=False)
print(f'Saved transfer log to {output_path}')


Saved transfer log to /Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data/diagnostics/wallet_0xbe90db_transfers.csv


## Notes

- Increase `MAX_TX_PER_WALLET` if you need deeper history (beware RPC rate limits).
- The `annotated` table shows links to any wallet already in `known_addresses.csv` (e.g., Binance or OKX deposit addresses).
- You can feed `transfers_df` into Gephi/Cytoscape for graph visualisation or join it with `cluster_sales_detail.csv` for time-series analysis.
